In [1]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from collections import defaultdict
import random
import os

seed = 3888
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

img_width, img_height = 100, 100
batch_size = 512
AUTOTUNE = tf.data.AUTOTUNE

tumour_folders = ["Invasive_Tumor", "DCIS_1", "DCIS_2", "Prolif_Invasive_Tumor"]
immune_folders = ["CD4+_T_Cells", "CD8+_T_Cells", "B_Cells", "Stromal"]
sub_label_list = tumour_folders + immune_folders
sub_label_to_index = {name: idx for idx, name in enumerate(sub_label_list)}

def get_file_label_pairs(folders, main_label):
    pairs = []
    for folder in folders:
        files = tf.io.gfile.glob(os.path.join(folder, "*.png"))
        for f in files:
            sub_label = os.path.basename(folder)
            pairs.append((f, main_label, sub_label))
    return pairs

tumour_files = get_file_label_pairs(tumour_folders, main_label=1)
immune_files = get_file_label_pairs(immune_folders, main_label=0)
all_file_label_pairs = tumour_files + immune_files
print(f"Total images before balancing: {len(all_file_label_pairs)}")

def create_balanced_equal_dataset(file_label_pairs, target_size=15000):
    label_to_files = defaultdict(list)
    for f, main_label, sub_label in file_label_pairs:
        label_to_files[sub_label].append((f, main_label, sub_label))

    balanced = []
    for sub_label, files in label_to_files.items():
        originals = files.copy()
        current_size = len(originals)

        if current_size < target_size:
            extra_needed = target_size - current_size
            oversampled_files = random.choices(originals, k=extra_needed)
            balanced.extend((f, m, s, False) for f, m, s in originals)
            balanced.extend((f, m, s, True) for f, m, s in oversampled_files)
        elif current_size > target_size:
            undersampled_files = random.sample(originals, k=target_size)
            balanced.extend((f, m, s, False) for f, m, s in undersampled_files)
        else:
            balanced.extend((f, m, s, False) for f, m, s in originals)

    random.shuffle(balanced)
    return balanced

target_size = 15000
balanced_pairs = create_balanced_equal_dataset(all_file_label_pairs, target_size = target_size)
print(f"Total images each class balances to: {target_size}")
print(f"Total images after balancing: {len(balanced_pairs)}")

def augment_image(img):
    angles = [0, 90, 180, 270]
    angle = random.choice(angles)
    
    if angle == 90:
        img = tf.image.rot90(img, k=1)
    elif angle == 180:
        img = tf.image.rot90(img, k=2)
    elif angle == 270:
        img = tf.image.rot90(img, k=3)

    flip_type = random.choice(['none', 'horizontal', 'vertical', 'both'])
    if flip_type == 'horizontal':
        img = tf.image.flip_left_right(img)
    elif flip_type == 'vertical':
        img = tf.image.flip_up_down(img)
    elif flip_type == 'both':
        img = tf.image.flip_left_right(img)
        img = tf.image.flip_up_down(img)

    return img

def decode_img(img_path, augment=False):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [img_width, img_height])
    img = img / 255.0
    if augment:
        img = augment_image(img)
    return img

def prepare_dataset(pairs):
    file_paths, main_labels, sub_labels, augment_flags = [], [], [], []
    
    for f, m, s, a in pairs:
        file_paths.append(f)
        main_labels.append(m)
        sub_labels.append(sub_label_to_index[s])
        augment_flags.append(a)

    file_paths = np.array(file_paths)
    main_labels = np.array(main_labels)
    sub_labels = np.array(sub_labels)
    augment_flags = np.array(augment_flags)

    train_files, test_files, train_main, test_main, train_sub, test_sub, train_aug, test_aug = train_test_split(
        file_paths, main_labels, sub_labels, augment_flags, test_size=0.2, stratify=sub_labels, random_state=seed
    )

    def process_path(file_path, main_label, sub_label, augment):
        img = decode_img(file_path, augment)
        return img, {"binary_output": main_label, "multiclass_output": sub_label}

    train_ds = tf.data.Dataset.from_tensor_slices((train_files, train_main, train_sub, train_aug))
    train_ds = train_ds.map(process_path, num_parallel_calls=AUTOTUNE).shuffle(1000).batch(batch_size).prefetch(AUTOTUNE)

    test_ds = tf.data.Dataset.from_tensor_slices((test_files, test_main, test_sub, test_aug))
    test_ds = test_ds.map(process_path, num_parallel_calls=AUTOTUNE).batch(batch_size).prefetch(AUTOTUNE)

    return train_ds, test_ds

train_ds_over, test_ds_over = prepare_dataset(balanced_pairs)

Total images before balancing: 124622
Total images each class balances to: 15000
Total images after balancing: 120000
